# Clase 9: Ingeniería de Datos y Clasificación de Billetes (PYG)

En esta sesión abandonamos los datasets industriales genéricos para construir nuestra propia inteligencia desde cero. El objetivo es desarrollar un **clasificador de billetes de Guaraníes** diseñado para una terminal de servicio automática.

---

## 0. Contexto de Ingeniería: ¿Por qué construimos nuestro propio Dataset?

En las clases anteriores (como con el dataset NEU), trabajamos con datos "curados": imágenes centradas, con iluminación perfecta y etiquetas validadas. En el mundo real, y específicamente en Paraguay, un **Sistema de Depósito Automático** se enfrenta a condiciones hostiles que el ingeniero debe prever.

### El Problema del "Mundo Real" vs. El Laboratorio
Un modelo de visión artificial para terminales de servicio falla no por falta de "neuronas", sino por falta de representatividad en los datos. Debemos considerar:
* **Variabilidad de Dispositivos:** No todos los alumnos tienen la misma cámara. Esto introduce diferentes niveles de ruido y balance de blancos.
* **Degradación del Medio:** Los billetes de baja denominación (2.000 y 5.000) suelen estar más desgastados, sucios o escritos que los de 100.000.
* **Condiciones de Iluminación:** La terminal puede estar bajo luz fluorescente, luz solar directa o en penumbra.

### El Concepto de "Data-Centric AI"
Estamos pasando de un enfoque basado en el modelo (donde solo tocamos la arquitectura) a un enfoque **Data-Centric**, donde la calidad y la diversidad del dataset son la prioridad. Si el modelo falla al reconocer un billete de 50.000 Gs. bajo una sombra, la solución no es una red más grande, sino mejores datos de entrenamiento que incluyan sombras.



---

## 1. El Desafío: Data Engineering (Recolección y Curación)

A diferencia de las clases anteriores, **no hay un link de descarga**. Como ingenieros de la FCyT, su primer paso es la fase de **Ingeniería de Campo**.

### Instrucciones Críticas para el Grupo:
1. **Estructura de Directorios:** Creen la carpeta raíz `dataset_billetes`. Dentro, separen estrictamente `train` (80%) y `val` (20%).
2. **Jerarquía de Clases:** Dentro de cada carpeta, creen subcarpetas por denominación: `2k`, `5k`, `10k`, `20k`, `50k`, `100k`. *Nota: Eviten espacios en los nombres de carpetas.*
3. **Protocolo de Captura (Variabilidad):** Capturen al menos 10 fotos por denominación bajo las siguientes condiciones:
    * **Fondo:** Cambien la superficie (madera del pupitre, piso, palma de la mano).
    * **Oclusión:** Tomen fotos donde el billete esté ligeramente doblado o tapado parcialmente por un dedo.
    * **Perspectiva:** No todas las fotos deben ser cenitales (desde arriba); inclinen el celular para simular la entrada a una ranura.



> **Regla de Oro de la Cátedra:** "Garbage In, Garbage Out". Si su recolección es descuidada, su terminal de servicios será un fracaso financiero.

## 2. Preparación del Laboratorio (Setup de Software y Hardware)
Antes de entrenar "el cerebro" de nuestra terminal de servicios, debemos preparar el entorno. Este bloque realiza tres tareas críticas:

Importación de Herramientas: Traemos las librerías de visión artificial (torchvision), procesamiento numérico (numpy) y manejo de imágenes (PIL).

Protocolo de Hardware: Detectamos si la RTX 5070 Ti está disponible. En ingeniería, siempre intentamos delegar el cálculo pesado a la GPU (CUDA) para reducir el tiempo de entrenamiento de horas a minutos.

Gestión de Dispositivo: Configuramos una variable device que usaremos en todo el proyecto para asegurar que los datos y el modelo vivan en el mismo componente físico.

In [ ]:
# =================================================================
# BLOQUE 1: CONFIGURACIÓN DEL ENTORNO Y DETECCIÓN DE HARDWARE
# =================================================================

import torch               # El motor principal de Deep Learning (PyTorch)
import torch.nn as nn        # Módulos para construir capas de la red neuronal
import torch.optim as optim  # Algoritmos de optimización (los "ajustadores" de pesos)
from torchvision import datasets, models, transforms # Herramientas específicas de Visión
from torch.utils.data import DataLoader              # El "gestor de logística" de los datos
import matplotlib.pyplot as plt                      # Para graficar resultados y ver billetes
import os                                            # Para navegar en las carpetas de Windows/Linux
import numpy as np                                   # Álgebra lineal y manejo de matrices
from PIL import Image                                # Librería estándar para abrir fotos

# --- PROTOCOLO DE ACELERACIÓN POR HARDWARE ---
# Verificamos si PyTorch puede ver la tarjeta de video (GPU) de la Predator.
# Si está disponible, usaremos 'cuda'. De lo contrario, usaremos el procesador (cpu).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mostramos por pantalla el dispositivo para que el alumno valide su configuración.
# ¡OJO!: Si sale 'cpu' y están en la Predator, algo anda mal con los drivers.
print(f">>> El modelo se ejecutará en: {device}")

# Si hay GPU, mostramos un dato extra de "salud del sistema"
if torch.cuda.is_available():
    print(f">>> GPU Detectada: {torch.cuda.get_device_name(0)}")

## Observación:
Es vital que entiendan la línea del device. En la Clase 9, cuando capturen las fotos de los billetes, esas fotos entran a la RAM (CPU), pero para que la red aprenda rápido, debemos "subirlas" a la VRAM (GPU). Si intentan operar un tensor que está en la CPU con un modelo que está en la GPU, el código lanzará un error de RuntimeError: Expected all tensors to be on the same device.

## 3. Data Augmentation: Multiplicando el conocimiento
En esta fase, aplicamos una técnica de Regularización. Como solo capturamos unas pocas fotos por billete, "engañamos" a la red neuronal mostrándole versiones modificadas de esas mismas imágenes en cada época de entrenamiento. Esto obliga a la red a no memorizar el billete, sino a aprender sus rasgos invariantes (el color, la posición de los próceres, los números).

¿Qué estamos simulando técnicamente?
Rotación y Recorte: El usuario nunca pone el billete perfectamente centrado en la ranura.

Color Jitter (Ruido de color): Diferentes cámaras y diferentes horas del día en el local.

Normalización: Ajustamos los valores de los píxeles a los promedios de ImageNet (el dataset con el que se entrenó originalmente MobileNetV2) para que el Transfer Learning sea efectivo.

In [ ]:
# =================================================================
# BLOQUE 3: PIPELINE DE TRANSFORMACIONES (DATA AUGMENTATION)
# =================================================================

# Definimos las medias y desviaciones estándar de ImageNet. 
# Esto es un estándar industrial: ayuda a que el modelo pre-entrenado
# "entienda" mejor nuestras fotos de Guaraníes.
norm_mean = [0.485, 0.456, 0.406]
norm_std = [0.229, 0.224, 0.225]

# Creamos un diccionario de transformaciones para Entrenamiento y Validación
data_transforms = {
    'train': transforms.Compose([
        # 1. Redimensionamos a 224x224 (tamaño de entrada de MobileNetV2)
        transforms.Resize((224, 224)),
        
        # 2. Rotación aleatoria: simula que el billete entró torcido (hasta 30 grados)
        transforms.RandomRotation(30),
        
        # 3. Recorte aleatorio: simula que el billete está cerca o lejos de la cámara
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        
        # 4. Espejo: el billete puede estar de frente o de espaldas
        transforms.RandomHorizontalFlip(p=0.5),
        
        # 5. Jitter: Variamos brillo y contraste (luces de oficina vs luz de sol)
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
        
        # 6. Convertimos a Tensor (formato matemático que entiende la RTX 5070 Ti)
        transforms.ToTensor(),
        
        # 7. Normalizamos para que los valores tengan media 0 y desviación 1
        transforms.Normalize(norm_mean, norm_std)
    ]),
    
    'val': transforms.Compose([
        # En validación NO aumentamos datos (queremos ver la realidad),
        # solo redimensionamos y normalizamos.
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(norm_mean, norm_std)
    ]),
}

print(">>> Pipeline de Aumentación configurado con éxito.")
print(">>> Nota: Se han incluido variaciones de luz para mayor robustez en la terminal.")

## Observación
Es fundamental que noten que en el conjunto de validación (val) no aplicamos rotaciones ni cambios de color aleatorios. ¿Por qué? Porque la validación debe representar la "verdad" de lo que la cámara captura. Si alteramos la validación, no sabríamos si el modelo realmente está funcionando bien en condiciones reales.


In [ ]:
# =================================================================
# BLOQUE 3.5: CARGA DINÁMICA DEL DATASET (EL PUENTE)
# =================================================================

data_dir = './dataset_billetes' # La carpeta que los chicos crearon

# Creamos los objetos Dataset (mapeo de carpetas)
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}

# Creamos los DataLoaders (los encargados de pasar los billetes a la GPU en grupos)
dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True)
              for x in ['train', 'val']}

# Guardamos los nombres de las clases (2k, 5k, etc.) y tamaños
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f">>> Clases detectadas: {class_names}")
print(f" >>> Imágenes de entrenamiento: {dataset_sizes['train']}")

## 4. Transfer Learning: No reinventar la rueda
En lugar de diseñar una red desde cero y esperar días a que aprenda a ver bordes y colores, usaremos Transfer Learning (Aprendizaje por Transferencia). Utilizaremos MobileNetV2, una arquitectura que ya fue entrenada por Google con más de 1.4 millones de imágenes (ImageNet).

¿Por qué MobileNetV2 para nuestra terminal?
* Eficiencia: Está diseñada para funcionar en celulares. En una terminal de servicios, queremos que el reconocimiento sea instantáneo.

* Conocimiento previo: La red ya sabe qué es una línea, un círculo y una textura. Nosotros solo le enseñaremos a distinguir si esas texturas pertenecen a un billete de 2.000 o de 100.000 Gs.

* Capas Congeladas: Bloquearemos las capas iniciales para que no cambien (ya son expertas en visión) y solo entrenaremos la parte final (el "clasificador").

In [ ]:
# =================================================================
# BLOQUE 4: ARQUITECTURA DEL MODELO (TRANSFER LEARNING)
# =================================================================

def crear_modelo_pyg(num_clases):
    # 1. Cargamos MobileNetV2 con sus "pesos" ya entrenados en ImageNet
    # Esto descarga el modelo experto de los servidores de PyTorch
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    
    # 2. CONGELAMIENTO (Freezing): 
    # Le decimos a PyTorch que NO calcule gradientes para estas capas.
    # No queremos que la red olvide cómo ver formas básicas.
    for param in model.parameters():
        param.requires_grad = False
    
    # 3. REDISEÑO DEL CLASIFICADOR:
    # MobileNetV2 termina originalmente en una capa que clasifica 1000 cosas.
    # Nosotros la reemplazamos por una que clasifique nuestras N denominaciones.
    
    # model.classifier[1] es la última capa densa de la red
    # 1280 es el número de "neuronas de entrada" que nos entrega MobileNet
    model.classifier[1] = nn.Sequential(
        nn.Linear(1280, 512),    # Capa intermedia de 512 neuronas
        nn.ReLU(),               # Activación para introducir no-linealidad
        nn.Dropout(0.3),         # Apagamos el 30% de neuronas para evitar Overfitting
        nn.Linear(512, num_clases) # Capa final: una neurona por cada billete
    )
    
    return model

# --- INSTANCIACIÓN ---
# Nota: num_clases se definirá automáticamente según las carpetas del dataset
print(">>> Función de arquitectura definida.")
print(">>> Estrategia: MobileNetV2 con extractor de características congelado.")

## Observación 
Es vital explicar que el Dropout(0.3) es como un "entrenamiento con pesas". Al apagar neuronas al azar, obligamos a las demás a aprender los rasgos del billete, evitando que la red se vuelva perezosa y dependa de un solo detalle (como una mancha específica en un billete de prueba).

In [ ]:
# =================================================================
# BLOQUE 4.5: CICLO DE ENTRENAMIENTO (TRAINING LOOP)
# =================================================================

def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        print(f'Época {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Modo entrenamiento
            else:
                model.eval()   # Modo evaluación

            running_loss = 0.0
            running_corrects = 0

            # Iterar sobre los datos
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad() # Limpiar gradientes

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward() # Backpropagation
                        optimizer.step() # Actualizar pesos

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.upper()} - Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    print('>>> Entrenamiento completado.')
    return model

## 5. Inferencia: El Momento de la Verdad (Simulación de Terminal)
En esta etapa, el modelo deja de aprender y pasa a modo evaluación (model.eval()). Simularemos el comportamiento de una ranura de depósito: el sistema recibe una imagen, la procesa a través de los mismos filtros que usamos en el entrenamiento y nos entrega una probabilidad para cada denominación de Guaraní.

Conceptos Clave en este Bloque:
* torch.no_grad(): Le decimos a la Predator que no gaste memoria calculando gradientes; solo queremos el resultado hacia adelante (Forward pass).

* Softmax: Convertimos las salidas "crudas" de la red en porcentajes (0 a 100%) para medir la confianza.

* Umbral de Seguridad: En una terminal real, si la confianza es menor al 80%, deberíamos rechazar el billete por seguridad.

In [ ]:
# =================================================================
# BLOQUE 5: MOTOR DE INFERENCIA PARA DEPÓSITO AUTOMÁTICO
# =================================================================

def procesar_deposito(ruta_imagen, model, clases):
    """
    Simula la entrada de un billete a la terminal de servicios.
    """
    # 1. Pasamos el modelo a modo evaluación (apaga Dropout)
    model.eval()
    
    # 2. Cargamos y preparamos la imagen
    img = Image.open(ruta_imagen).convert('RGB')
    
    # Aplicamos las transformaciones de validación (Resize + Normalización)
    # Unsqueeze(0) añade la dimensión del 'Batch' -> [1, 3, 224, 224]
    img_t = data_transforms['val'](img).unsqueeze(0).to(device)
    
    # 3. Realizamos la predicción sin calcular gradientes (más rápido)
    with torch.no_grad():
        outputs = model(img_t)
        
        # Aplicamos Softmax para obtener probabilidades legibles
        probs = torch.nn.functional.softmax(outputs, dim=1)
        
        # Obtenemos la confianza más alta y el índice de la clase
        confianza, prediccion = torch.max(probs, 1)
        
    # 4. RESULTADO EN PANTALLA
    clase_detectada = clases[prediccion.item()]
    porcentaje = confianza.item() * 100
    
    print("-" * 40)
    print(f" LOG DE TERMINAL: ")
    print(f" >>> OBJETO DETECTADO: Billete de {clase_detectada} PYG")
    print(f" >>> NIVEL DE CONFIANZA: {porcentaje:.2f}%")
    print("-" * 40)
    
    # Visualización para el ingeniero
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title(f"Predicción: {clase_detectada} PYG ({porcentaje:.2f}%)")
    plt.axis('off')
    plt.show()

# --- NOTA PARA EL ALUMNO ---
# Para probar este bloque, asegúrense de tener una imagen de prueba 
# en su computadora y llamen a la función así:
# procesar_deposito('ruta/de/tu/foto.jpg', model, train_set.classes)

## Observación:
Fíjense en el valor de la Confianza. Si el modelo dice que es un billete de 100.000 Gs. pero con un 55% de confianza, ¡cuidado! Como ingenieros, debemos programar una lógica de negocio que diga: "Si confianza < 85%, devolver billete al usuario". La IA no es infalible, y nuestro trabajo es poner los límites de seguridad.

# Del Modelo al Producto Final (Despliegue)

Un modelo de Inteligencia Artificial no tiene valor si se queda viviendo solo dentro de un notebook. En ingeniería, el paso final es el **Despliegue** (*Deployment*). Para nuestra terminal de servicios, esto implica dos pasos críticos: 
1. **Serialización:** Guardar el modelo entrenado en un archivo físico.
2. **Interfaz de Inferencia:** Crear una lógica que reciba una imagen nueva y entregue una respuesta en milisegundos.

---

## 6. Exportación y Simulación de Terminal

En esta celda, primero guardaremos los "pesos" que la red aprendió tras el entrenamiento. Luego, utilizaremos una interfaz sencilla para procesar depósitos de billetes en tiempo real.



### Tips y Pasos Previos para el Éxito:
* **Umbral de Decisión (Threshold):** En una terminal real, nunca aceptamos un billete con menos del 85% de confianza. Es preferible pedirle al usuario que "reintente" a procesar un billete erróneo.
* **Gestión de Memoria:** Al exportar el modelo con `.pth`, solo guardamos los parámetros numéricos. Para volver a usarlo en otra App, necesitamos tener definida la arquitectura original.
* **Calibración:** Si el modelo confunde mucho el de 2.000 con el de 5.000, el ingeniero debe volver al Paso #1 y recolectar fotos con fondos más diversos.

In [ ]:
# =================================================================
# BLOQUE 6: GUARDADO Y PROTOTIPO DE TERMINAL DE SERVICIO
# =================================================================

# --- PASO A: GUARDAR EL MODELO (CHECKPOINT) ---
# Guardamos solo los 'state_dict' (los pesos aprendidos). 
# Es la forma más eficiente y profesional de exportar en PyTorch.
nombre_archivo = 'modelo_terminal_guaranies.pth'
torch.save(model.state_dict(), nombre_archivo)
print(f">>> ÉXITO: Modelo exportado como '{nombre_archivo}'")
print(>>> "Tip: Este archivo es el que subirías a la memoria de la terminal real.")

# --- PASO B: INTERFAZ DE PROCESAMIENTO DE DEPÓSITOS ---

def terminal_servicio_pyg(ruta_imagen, model_path, clases):
    """
    Simula el software de una terminal de servicios de la FCyT.
    """
    # 1. Reconstruimos la arquitectura y cargamos los pesos guardados
    # Nota: Usamos la función crear_modelo_pyg definida en el Bloque 4
    modelo_inferencia = crear_modelo_pyg(len(clases))
    modelo_inferencia.load_state_dict(torch.load(model_path))
    modelo_inferencia.to(device)
    modelo_inferencia.eval()
    
    # 2. Procesamos la imagen del billete
    img = Image.open(ruta_imagen).convert('RGB')
    img_t = data_transforms['val'](img).unsqueeze(0).to(device)
    
    # 3. Inferencia de alta velocidad
    with torch.no_grad():
        outputs = modelo_inferencia(img_t)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confianza, prediccion = torch.max(probs, 1)
    
    # 4. LÓGICA DE NEGOCIO DE LA TERMINAL
    valor_detectado = clases[prediccion.item()]
    score = confianza.item() * 100
    UMBRAL_SEGURIDAD = 85.0 # Porcentaje mínimo para aceptar el dinero
    
    print("\n" + "="*45)
    print("       SISTEMA DE DEPÓSITO AUTOMÁTICO - UNCA")
    print("="*45)
    
    if score >= UMBRAL_SEGURIDAD:
        print(f" RESULTADO: Billete de {valor_detectado} PYG")
        print(f" ESTADO: DEPÓSITO ACEPTADO ✅")
        print(f" CONFIANZA: {score:.2f}%")
    else:
        print(f" RESULTADO: No se puede validar con certeza")
        print(f" ESTADO: DEPÓSITO RECHAZADO ❌")
        print(f" MOTIVO: Confianza insuficiente ({score:.2f}%)")
        print(f" ACCIÓN: Por favor, intente de nuevo o use otro billete.")
    
    print("="*45)
    
    # Visualización del billete ingresado
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# --- EJEMPLO DE USO ---
# Una vez que los alumnos tengan su modelo entrenado y una foto de prueba:
# terminal_servicio_pyg('mi_billete_prueba.jpg', nombre_archivo, train_set.classes)